# Detección Automática de Zonas Deforestadas en Colombia
### Imágenes Satelitales y Deep Learning como Herramienta para Política Pública

**Curso:** Deep Learning — Maestría en Inteligencia Analítica de Datos (MIAD)  
**Repositorio:** [Andres-Acuna/dl_deforestation](https://github.com/Andres-Acuna/dl_deforestation)

---

Este notebook detecta automáticamente el entorno de ejecución (local o Colab) y configura las rutas correspondientes. El código es idéntico para todos los integrantes del equipo independientemente del entorno.

| Sección | Descripción |
|---------|-------------|
| 0 | Configuración del entorno y variables globales |
| 1 | Dataset Planet Amazon |
| 2 | Imágenes Sentinel-2 — Google Earth Engine |
| 3 | Rasters del IDEAM — máscaras de deforestación |
| 4 | Pipeline de datos |
| 5 | Arquitectura del modelo |
| 6 | Entrenamiento Fase 1 — Planet Amazon |
| 7 | Fine-tuning Fase 2 — Colombia |
| 8 | Evaluación y resultados |
| 9 | Prototipo de aplicación |


---
## 0. Configuración del Entorno

La siguiente celda detecta automáticamente si se está ejecutando en Google Colab o en un entorno local, y configura las rutas del proyecto en consecuencia.

> **Para agregar un nuevo integrante local:** añadir una entrada al diccionario `RUTAS_LOCALES` con el nombre de usuario del sistema operativo y la ruta raíz del proyecto, luego hacer push.

In [ ]:
import os, sys, subprocess, getpass
from pathlib import Path

# ── Detección de entorno ─────────────────────────────────────────────
EN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content/drive')

if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE = Path('/content/drive/MyDrive/deforestation_project')
    REPO = Path('/content/dl_deforestation')

    if REPO.exists():
        r = subprocess.run(['git', 'pull'], cwd=REPO, capture_output=True, text=True)
        print(f'Repositorio actualizado: {r.stdout.strip()}')
    else:
        subprocess.run(['git', 'clone',
                         'https://github.com/Andres-Acuna/dl_deforestation.git',
                         str(REPO)], capture_output=True)
        print(f'Repositorio clonado en {REPO}')

    if str(REPO) not in sys.path:
        sys.path.insert(0, str(REPO))

else:
    # ── Rutas por integrante ─────────────────────────────────────────
    # Agregar una entrada por cada integrante con su usuario de SO
    RUTAS_LOCALES = {
        'camil'  : Path(r'C:\Users\camil\Documentos\MIAD\Deep Learning\Proyecto'),
        'andres' : Path(r'C:\Users\andres\Documents\dl_deforestation'),
        # 'nuevo_integrante': Path(r'ruta\al\proyecto'),
    }

    usuario = getpass.getuser().lower()
    BASE    = None

    for nombre, ruta in RUTAS_LOCALES.items():
        if nombre in usuario or ruta.exists():
            BASE = ruta
            break

    if BASE is None:
        raise EnvironmentError(
            f'Usuario "{usuario}" no reconocido en RUTAS_LOCALES.\n'
            'Agrega tu entrada al diccionario y haz push antes de continuar.'
        )

assert BASE.exists(), f'La carpeta base no existe: {BASE}'

print(f'Entorno  : {"Colab" if EN_COLAB else "Local"}')
print(f'Usuario  : {getpass.getuser()}')
print(f'Base     : {BASE}')


In [ ]:
import torch
import numpy as np

# ── Rutas derivadas de BASE ──────────────────────────────────────
DATA_DIR      = BASE / 'data'
RAW_PLANET    = DATA_DIR / 'raw' / 'planet'
RAW_COLOMBIA  = DATA_DIR / 'raw' / 'colombia'
PROC_PLANET   = DATA_DIR / 'processed' / 'planet'
PROC_COLOMBIA = DATA_DIR / 'processed' / 'colombia'
MASKS_DIR     = DATA_DIR / 'masks'
CKPT_DIR      = BASE / 'checkpoints'
LOGS_DIR      = BASE / 'logs'
OUTPUTS_DIR   = BASE / 'outputs'
PLANET_CSV    = RAW_PLANET / 'train_v2.csv'
PLANET_TIF    = RAW_PLANET / 'train-tif-v2'
BEST_CKPT     = CKPT_DIR / 'planet_best.pt'
COL_CKPT      = CKPT_DIR / 'colombia_best.pt'

for d in [PROC_PLANET, PROC_COLOMBIA, MASKS_DIR, CKPT_DIR, LOGS_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Hiperparámetros ──────────────────────────────────────────────
CONFIG = {
    'img_size'       : 224,
    'batch_size'     : 32,
    # En Windows con DataLoader, num_workers > 0 puede causar errores.
    # Si ocurre RuntimeError al crear el DataLoader, cambiar a 0.
    'num_workers'    : 0 if os.name == 'nt' else 4,
    'tile_size'      : 224,
    'tile_stride'    : 112,
    'planet_epochs'  : 20,
    'planet_lr'      : 1e-3,
    'num_classes'    : 17,
    'colombia_epochs': 30,
    'colombia_lr'    : 5e-4,
    'seed'           : 42,
}

PLANET_TAGS = [
    'agriculture', 'artisinal_mine', 'bare_ground', 'blooming',
    'blow_down', 'clear', 'cloudy', 'conventional_mine', 'cultivation',
    'habitation', 'haze', 'partly_cloudy', 'primary', 'road',
    'selective_logging', 'slash_burn', 'water'
]
DEFORESTATION_TAGS = {
    'agriculture', 'slash_burn', 'habitation', 'cultivation',
    'bare_ground', 'artisinal_mine', 'selective_logging', 'blow_down'
}

torch.manual_seed(CONFIG['seed'])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Verificación de rutas ─────────────────────────────────────────
print('Rutas del proyecto:')
for nombre, ruta in [
    ('Planet raw   ', RAW_PLANET),
    ('Colombia raw ', RAW_COLOMBIA),
    ('Mascaras     ', MASKS_DIR),
    ('Checkpoints  ', CKPT_DIR),
    ('Outputs      ', OUTPUTS_DIR),
]:
    print(f'  [{"OK" if ruta.exists() else "NO ENCONTRADA"}] {nombre} {ruta}')

print(f'\nDispositivo  : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
    print(f'VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'CUDA version : {torch.version.cuda}')


---
## 1. Dataset Planet Amazon

El dataset contiene aproximadamente 40.000 imágenes multiespectrales de cuatro bandas (Azul, Verde, Rojo, NIR) de 256×256 píxeles sobre el Amazonas brasileño, etiquetadas con múltiples clases de cobertura terrestre.

Sirve como fuente de preentrenamiento del backbone: la red aprende representaciones espectrales de regiones forestales y perturbadas antes de transferirse a Colombia.

### 1.1 Descompresión

In [ ]:
import zipfile

def contar_imagenes(directorio):
    return (len(list(Path(directorio).rglob('*.tif'))) +
            len(list(Path(directorio).rglob('*.jpg'))))

n_existentes = contar_imagenes(RAW_PLANET)

if n_existentes > 0:
    print(f'Dataset ya descomprimido ({n_existentes:,} imagenes). Se omite este paso.')

else:
    zips = list(RAW_PLANET.glob('*.zip'))
    if len(zips) == 0:
        print(f'No se encontro ningun ZIP en {RAW_PLANET}')
        print('Descarga el dataset con:')
        print('  kaggle datasets download -d nikitarom/planets-dataset -p data/raw/planet/')
    else:
        zip_path = zips[0]
        print(f'Descomprimiendo {zip_path.name} ({zip_path.stat().st_size/1e9:.2f} GB)...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            archivos = z.namelist()
            total = len(archivos)
            for i, archivo in enumerate(archivos):
                z.extract(archivo, RAW_PLANET)
                if i % 5000 == 0 and i > 0:
                    print(f'  {i:,}/{total:,} ({i/total*100:.1f}%)')
        print(f'Listo: {contar_imagenes(RAW_PLANET):,} imagenes extraidas.')


### 1.2 Exploración de etiquetas

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

csv_candidatos = list(RAW_PLANET.rglob('train_v2.csv'))
assert len(csv_candidatos) > 0, (
    'No se encontro train_v2.csv. Verifica la descompresion.'
)
CSV_PATH = csv_candidatos[0]

df = pd.read_csv(CSV_PATH)
df['tags_list'] = df['tags'].str.split()
df['es_deforestada'] = df['tags_list'].apply(
    lambda t: int(bool(set(t) & DEFORESTATION_TAGS))
)

print(f'Total imagenes    : {len(df):,}')
print(f'Con deforestacion : {df["es_deforestada"].sum():,} ({df["es_deforestada"].mean():.1%})')
display(df.head())


In [ ]:
all_tags = [t for tags in df['tags_list'] for t in tags]
conteo   = Counter(all_tags)
tags_ord = sorted(conteo.items(), key=lambda x: -x[1])

fig, ax = plt.subplots(figsize=(10, 5))
nombres = [t[0] for t in tags_ord]
valores = [t[1] for t in tags_ord]
colores = ['#c0392b' if n in DEFORESTATION_TAGS else '#2e7d32' for n in nombres]
ax.barh(nombres, valores, color=colores)
ax.set_xlabel('Frecuencia')
ax.set_title('Distribucion de etiquetas — Planet Amazon')
ax.invert_yaxis()
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#c0392b', label='Relacionada con deforestacion'),
    Patch(color='#2e7d32', label='Otras clases')
], loc='lower right')
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'distribucion_etiquetas.png', dpi=120)
plt.show()


### 1.3 Exploración visual

In [ ]:
import rasterio

tif_candidatos = list(RAW_PLANET.rglob('train-tif-v2'))
assert len(tif_candidatos) > 0, 'No se encontro la carpeta train-tif-v2.'
TIF_DIR = tif_candidatos[0]

muestra = sorted(TIF_DIR.glob('*.tif'))[0]
with rasterio.open(muestra) as src:
    print(f'Bandas    : {src.count}  (esperado: 4)')
    print(f'Dimension : {src.height} x {src.width} px')
    print(f'Tipo dato : {src.dtypes[0]}')
    for i, nombre in enumerate(['Azul','Verde','Rojo','NIR'], start=1):
        b = src.read(i).astype(np.float32)
        print(f'  Banda {i} {nombre:<6} '
              f'min={b.min():.0f}  max={b.max():.0f}  media={b.mean():.0f}')


In [ ]:
def normalizar(banda):
    p2, p98 = np.percentile(banda, [2, 98])
    return np.clip((banda - p2) / (p98 - p2 + 1e-6), 0, 1)

def leer_rgb(ruta):
    with rasterio.open(ruta) as src:
        r = normalizar(src.read(3).astype(np.float32))
        g = normalizar(src.read(2).astype(np.float32))
        b = normalizar(src.read(1).astype(np.float32))
    return np.stack([r, g, b], axis=-1)

clases_muestra = ['primary','agriculture','slash_burn','habitation','bare_ground','water']
muestras = []
for clase in clases_muestra:
    filas = df[df['tags_list'].apply(lambda t: clase in t)]
    if len(filas) > 0:
        ruta = TIF_DIR / f'{filas.iloc[0]["image_name"]}.tif'
        if ruta.exists():
            muestras.append((clase, ruta))

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
fig.suptitle('Muestras del dataset — color verdadero (RGB)', fontsize=13)
for ax, (clase, ruta) in zip(axes.flat, muestras):
    ax.imshow(leer_rgb(ruta))
    color = '#c0392b' if clase in DEFORESTATION_TAGS else '#2e7d32'
    ax.set_title(clase, fontsize=10, color=color, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'muestras_planet.png', dpi=120)
plt.show()


---
## 2. Imágenes Sentinel-2 para Colombia (Google Earth Engine)

Exportación de composiciones medianas anuales de Sentinel-2 a Google Drive. Una vez que las tareas de GEE terminan, los archivos deben descargarse manualmente a `data/raw/colombia/`.

> Requiere cuenta de GEE activa: https://earthengine.google.com/signup/  
> Monitoreo de tareas exportadas: https://code.earthengine.google.com/tasks

In [ ]:
import ee

try:
    ee.Initialize()
    print('Earth Engine inicializado.')
except Exception:
    ee.Authenticate()
    ee.Initialize()
    print('Autenticacion completada.')


In [ ]:
def enmascarar_nubes_s2(imagen):
    scl = imagen.select('SCL')
    return imagen.updateMask(
        scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    )

def exportar_sentinel2(nombre_aoi, coordenadas, anio, carpeta_drive):
    aoi = ee.Geometry.Polygon([coordenadas])
    imagen = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterDate(f'{anio}-01-01', f'{anio}-12-31')
        .filterBounds(aoi)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .map(enmascarar_nubes_s2)
        .select(['B2','B3','B4','B8'])
        .median()
        .clip(aoi)
    )
    tarea = ee.batch.Export.image.toDrive(
        image=imagen,
        description=f's2_{nombre_aoi}_{anio}',
        folder=carpeta_drive,
        scale=10, region=aoi, maxPixels=1e10,
        fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': True}
    )
    tarea.start()
    return tarea

AREAS_INTERES = {
    'caqueta' : [[-76.5,0.5],[-73.0,0.5],[-73.0,2.5],[-76.5,2.5]],
    'meta'    : [[-74.9,2.9],[-72.0,2.9],[-72.0,4.5],[-74.9,4.5]],
    'putumayo': [[-77.0,0.0],[-74.5,0.0],[-74.5,1.5],[-77.0,1.5]],
}

tifs_colombia = list(RAW_COLOMBIA.rglob('*.tif'))

if len(tifs_colombia) > 0:
    print(f'Imagenes colombianas ya presentes ({len(tifs_colombia)}). Se omite la exportacion.')
    for f in tifs_colombia:
        print(f'  {f.name}  ({f.stat().st_size/1e6:.1f} MB)')
else:
    print('Iniciando exportacion a Google Drive...')
    for nombre, coords in AREAS_INTERES.items():
        exportar_sentinel2(nombre, coords, 2022,
                           'deforestation_project/data/raw/colombia')
        print(f'  Tarea iniciada: s2_{nombre}_2022')
    print(f'\nCuando terminen, descarga los GeoTIFF a:\n  {RAW_COLOMBIA}')


---
## 3. Rasters del IDEAM — Máscaras de Deforestación

Los rasters del IDEAM se reproyectan al mismo CRS y resolución que las imágenes Sentinel-2 para generar las máscaras binarias de entrenamiento.

Descarga el producto **Mapa de Pérdida de Bosque** desde:  
http://smbyc.ideam.gov.co/MonitoreoBC-WEB/  
y coloca los archivos `.tif` en `data/masks/`.

In [ ]:
from rasterio.warp import reproject, Resampling

def alinear_mascara(ruta_mascara, ruta_imagen_ref, ruta_salida):
    with rasterio.open(ruta_imagen_ref) as ref:
        crs_dst = ref.crs
        tf_dst  = ref.transform
        W, H    = ref.width, ref.height

    with rasterio.open(ruta_mascara) as src:
        datos = np.zeros((1, H, W), dtype=np.uint8)
        reproject(
            source=rasterio.band(src, 1),
            destination=datos,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=tf_dst,
            dst_crs=crs_dst,
            resampling=Resampling.nearest
        )

    datos_bin = (datos > 0).astype(np.uint8)
    with rasterio.open(ruta_salida, 'w', driver='GTiff',
                       height=H, width=W, count=1,
                       dtype=np.uint8, crs=crs_dst, transform=tf_dst) as dst:
        dst.write(datos_bin)

    area_ha = datos_bin.sum() * 100 / 10_000
    print(f'Mascara guardada : {ruta_salida.name}')
    print(f'Area deforestada : {area_ha:,.0f} ha aprox.')


mascaras_raw = (list(MASKS_DIR.rglob('*ideam*.tif')) +
                list(MASKS_DIR.rglob('*bosque*.tif')) +
                list(MASKS_DIR.rglob('*deforest*.tif')))
imagenes_col = list(RAW_COLOMBIA.rglob('s2_*.tif'))

if len(mascaras_raw) == 0:
    print(f'Sin rasters IDEAM en {MASKS_DIR}.')
    print('Descarga el producto y coloca los archivos .tif en esa carpeta.')
elif len(imagenes_col) == 0:
    print('Primero completa la Seccion 2 (imagenes Sentinel-2).')
else:
    for mascara in mascaras_raw:
        salida = MASKS_DIR / f'{mascara.stem}_alineada.tif'
        if salida.exists():
            print(f'Ya procesada: {salida.name}')
        else:
            refs = [i for i in imagenes_col
                    if mascara.stem.split('_')[1] in i.stem]
            if refs:
                alinear_mascara(mascara, refs[0], salida)


---
## 4. Pipeline de Datos

Definición de datasets y dataloaders para las dos fases del entrenamiento. El dataset Planet se usa para clasificación multi-label. El dataset Colombia, construido a partir de tiles generados sobre Sentinel-2, se usa para segmentación binaria píxel a píxel.

### 4.1 Dataset Planet Amazon

In [ ]:
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

TAG_TO_IDX = {tag: i for i, tag in enumerate(PLANET_TAGS)}

def get_transforms_planet(modo='train'):
    comunes = [
        A.Normalize(mean=[0.485,0.456,0.406,0.5],
                    std=[0.229,0.224,0.225,0.25]),
        ToTensorV2()
    ]
    if modo == 'train':
        return A.Compose([
            A.RandomCrop(CONFIG['img_size'], CONFIG['img_size']),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.OneOf([A.GaussNoise(p=1), A.GaussianBlur(p=1)], p=0.2),
            A.ColorJitter(brightness=0.2, contrast=0.2, p=0.3),
        ] + comunes)
    return A.Compose(
        [A.CenterCrop(CONFIG['img_size'], CONFIG['img_size'])] + comunes
    )


class PlanetDataset(Dataset):
    def __init__(self, df, tif_dir, modo='train'):
        n = len(df)
        cortes = {'train': (0, int(n*0.8)),
                  'val':   (int(n*0.8), int(n*0.9)),
                  'test':  (int(n*0.9), n)}
        a, b = cortes[modo]
        self.df        = df.iloc[a:b].reset_index(drop=True)
        self.tif_dir   = Path(tif_dir)
        self.transform = get_transforms_planet(modo)

    def _codificar(self, tags):
        vec = np.zeros(len(PLANET_TAGS), dtype=np.float32)
        for t in tags:
            if t in TAG_TO_IDX:
                vec[TAG_TO_IDX[t]] = 1.0
        return vec

    def __len__(self):  return len(self.df)

    def __getitem__(self, idx):
        import torch
        fila = self.df.iloc[idx]
        with rasterio.open(self.tif_dir / f'{fila["image_name"]}.tif') as src:
            img = src.read([1,2,3,4]).astype(np.float32)
        img = np.transpose(np.clip(img / 10000.0, 0, 1), (1,2,0))
        aug = self.transform(image=img)
        return aug['image'], torch.tensor(self._codificar(fila['tags_list']))


In [ ]:
# En Windows el DataLoader con num_workers > 0 requiere el guard
# 'if __name__ == __main__'. Como estamos en un notebook usamos 0.
if __name__ == '__main__' or True:
    ds_train = PlanetDataset(df, TIF_DIR, 'train')
    ds_val   = PlanetDataset(df, TIF_DIR, 'val')
    ds_test  = PlanetDataset(df, TIF_DIR, 'test')

    dl_train = DataLoader(ds_train, batch_size=CONFIG['batch_size'],
                          shuffle=True,  num_workers=CONFIG['num_workers'],
                          pin_memory=torch.cuda.is_available())
    dl_val   = DataLoader(ds_val,   batch_size=CONFIG['batch_size'],
                          shuffle=False, num_workers=CONFIG['num_workers'],
                          pin_memory=torch.cuda.is_available())
    dl_test  = DataLoader(ds_test,  batch_size=CONFIG['batch_size'],
                          shuffle=False, num_workers=CONFIG['num_workers'],
                          pin_memory=torch.cuda.is_available())

    print(f'Train : {len(ds_train):,}  ({len(dl_train)} batches)')
    print(f'Val   : {len(ds_val):,}  ({len(dl_val)} batches)')
    print(f'Test  : {len(ds_test):,}  ({len(dl_test)} batches)')

    imgs, labels = next(iter(dl_train))
    print(f'\nBatch de verificacion:')
    print(f'  Imagenes  : {imgs.shape}  dtype={imgs.dtype}')
    print(f'  Etiquetas : {labels.shape}  dtype={labels.dtype}')


### 4.2 Generación de tiles — Colombia

In [ ]:
def generar_tiles(ruta_imagen, ruta_mascara, dir_salida,
                  tam_tile=224, stride=112, min_validos=0.8):
    dir_salida = Path(dir_salida)
    (dir_salida / 'images').mkdir(parents=True, exist_ok=True)
    (dir_salida / 'masks').mkdir(parents=True, exist_ok=True)

    with rasterio.open(ruta_imagen) as img_src, \
         rasterio.open(ruta_mascara) as mask_src:
        H, W = img_src.height, img_src.width
        n_ok, n_skip = 0, 0
        for fila in range(0, H - tam_tile, stride):
            for col in range(0, W - tam_tile, stride):
                w = rasterio.windows.Window(col, fila, tam_tile, tam_tile)
                ti = img_src.read(window=w).astype(np.float32)
                tm = mask_src.read(1, window=w).astype(np.uint8)
                if np.isfinite(ti).all(axis=0).mean() < min_validos:
                    n_skip += 1
                    continue
                np.save(dir_salida / 'images' / f'tile_{n_ok:05d}.npy', ti)
                np.save(dir_salida / 'masks'  / f'tile_{n_ok:05d}.npy', tm)
                n_ok += 1

    print(f'  Generados: {n_ok:,}  |  Descartados: {n_skip:,}')
    return n_ok


imagenes_col = sorted(RAW_COLOMBIA.rglob('s2_*.tif'))
mascaras_col = sorted(MASKS_DIR.rglob('*_alineada.tif'))

if len(imagenes_col) == 0 or len(mascaras_col) == 0:
    print('Pendiente: completar Secciones 2 y 3 antes de generar tiles.')
else:
    tiles_existentes = list((PROC_COLOMBIA / 'images').rglob('*.npy'))
    if len(tiles_existentes) > 0:
        print(f'Tiles ya generados ({len(tiles_existentes):,}). Se omite este paso.')
    else:
        for img_p, mask_p in zip(imagenes_col, mascaras_col):
            print(f'Procesando: {img_p.name}')
            generar_tiles(img_p, mask_p, PROC_COLOMBIA,
                          CONFIG['tile_size'], CONFIG['tile_stride'])


### 4.3 Dataset Colombia

In [ ]:
class ColombiaDataset(Dataset):
    def __init__(self, tiles_dir, modo='train'):
        rutas = sorted((Path(tiles_dir) / 'images').glob('*.npy'))
        n = len(rutas)
        cortes = {'train': (0, int(n*0.70)),
                  'val':   (int(n*0.70), int(n*0.85)),
                  'test':  (int(n*0.85), n)}
        a, b = cortes[modo]
        self.rutas = rutas[a:b]
        comunes = [
            A.Normalize(mean=[0.485,0.456,0.406,0.5],
                        std=[0.229,0.224,0.225,0.25]),
            ToTensorV2()
        ]
        self.transform = A.Compose(
            ([A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
              A.RandomRotate90(p=0.5),
              A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, p=0.3)]
             if modo == 'train' else []) + comunes
        )

    def __len__(self):  return len(self.rutas)

    def __getitem__(self, idx):
        import torch
        ruta_img  = self.rutas[idx]
        ruta_mask = ruta_img.parent.parent / 'masks' / ruta_img.name
        img  = np.clip(np.load(ruta_img) / 10000.0, 0, 1)
        mask = np.load(ruta_mask)
        aug  = self.transform(image=np.transpose(img,(1,2,0)),
                              mask=mask.astype(np.float32))
        return aug['image'], aug['mask'].long()


---
## 5. Arquitectura del Modelo

EfficientNet-B4 como backbone adaptado a 4 canales de entrada. En la Fase 1 se acopla un head de clasificación multi-label. En la Fase 2 el backbone se transfiere como encoder de una U-Net para segmentación píxel a píxel.

La primera capa convolucional se amplía de 3 a 4 canales. Los pesos RGB se cargan desde ImageNet y el canal NIR se inicializa como el promedio de los otros tres, práctica estándar en modelos de teledetección.

In [ ]:
import torch.nn as nn
import torchvision.models as tv_models

class BackboneDeforestacion(nn.Module):
    def __init__(self, pretrained=True, canales=4):
        super().__init__()
        weights = tv_models.EfficientNet_B4_Weights.DEFAULT if pretrained else None
        base    = tv_models.efficientnet_b4(weights=weights)
        conv0   = base.features[0][0]
        nueva   = nn.Conv2d(canales, conv0.out_channels,
                            kernel_size=conv0.kernel_size,
                            stride=conv0.stride,
                            padding=conv0.padding,
                            bias=conv0.bias is not None)
        if pretrained:
            with torch.no_grad():
                nueva.weight[:, :3] = conv0.weight
                nueva.weight[:, 3]  = conv0.weight.mean(dim=1)
        base.features[0][0] = nueva
        self.features    = base.features
        self.avgpool     = base.avgpool
        self.feature_dim = base.classifier[1].in_features

    def forward(self, x):
        return torch.flatten(self.avgpool(self.features(x)), 1)


class ModeloPlanet(nn.Module):
    def __init__(self, pretrained=True, dropout=0.3):
        super().__init__()
        self.backbone = BackboneDeforestacion(pretrained=pretrained)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.backbone.feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(512, CONFIG['num_classes'])
        )

    def forward(self, x):  return self.head(self.backbone(x))


modelo_planet = ModeloPlanet(pretrained=True).to(DEVICE)
with torch.no_grad():
    x_test = torch.randn(2, 4, CONFIG['img_size'], CONFIG['img_size']).to(DEVICE)
    out    = modelo_planet(x_test)
n_params = sum(p.numel() for p in modelo_planet.parameters() if p.requires_grad)
print(f'Entrada            : {x_test.shape}')
print(f'Salida             : {out.shape}  (esperado: [2, {CONFIG["num_classes"]}])')
print(f'Parametros         : {n_params:,}')


---
## 6. Entrenamiento — Fase 1: Planet Amazon

Entrenamiento del backbone sobre el dataset Planet Amazon para clasificación multi-label. Se usa `BCEWithLogitsLoss` con pesos positivos para compensar el desbalance de clases y `OneCycleLR` como scheduler.

La métrica principal es el **F2-score por muestra**, que penaliza más los falsos negativos, relevante en un contexto donde omitir una zona deforestada tiene mayor costo que una falsa alarma.

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.metrics import fbeta_score

def entrenar_planet(modelo, dl_train, dl_val, config, ruta_ckpt):
    pos_w    = torch.ones(config['num_classes']) * 2.0
    criterio = nn.BCEWithLogitsLoss(pos_weight=pos_w.to(DEVICE))
    opt      = AdamW(modelo.parameters(),
                     lr=config['planet_lr'], weight_decay=1e-4)
    sched    = OneCycleLR(opt, max_lr=config['planet_lr'],
                          epochs=config['planet_epochs'],
                          steps_per_epoch=len(dl_train))
    mejor_f2  = 0.0
    historial = {'train_loss': [], 'val_f2': []}

    for epoca in range(config['planet_epochs']):
        modelo.train()
        loss_total = 0.0
        for imgs, labels in dl_train:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad()
            loss = criterio(modelo(imgs), labels)
            loss.backward()
            nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
            opt.step()
            sched.step()
            loss_total += loss.item()

        modelo.eval()
        preds_all, labels_all = [], []
        with torch.no_grad():
            for imgs, labels in dl_val:
                preds_all.append(torch.sigmoid(modelo(imgs.to(DEVICE))).cpu().numpy())
                labels_all.append(labels.numpy())

        preds  = (np.vstack(preds_all) > 0.2).astype(int)
        labels = np.vstack(labels_all)
        f2     = fbeta_score(labels, preds, beta=2,
                             average='samples', zero_division=0)
        loss_m = loss_total / len(dl_train)
        historial['train_loss'].append(loss_m)
        historial['val_f2'].append(f2)
        print(f'Epoca {epoca+1:>3}/{config["planet_epochs"]}  '
              f'loss={loss_m:.4f}  F2={f2:.4f}')

        if f2 > mejor_f2:
            mejor_f2 = f2
            torch.save({'epoch': epoca,
                        'model_state_dict': modelo.state_dict(),
                        'f2': mejor_f2}, ruta_ckpt)
            print(f'  Checkpoint guardado (F2={mejor_f2:.4f})')

    return historial


In [ ]:
if BEST_CKPT.exists():
    ckpt = torch.load(BEST_CKPT, map_location=DEVICE)
    modelo_planet.load_state_dict(ckpt['model_state_dict'])
    print(f'Checkpoint cargado — F2={ckpt["f2"]:.4f}  epoca={ckpt["epoch"]+1}')
    print('Para re-entrenar, elimina el archivo:', BEST_CKPT)
    historial_planet = None
else:
    print(f'Iniciando entrenamiento en {DEVICE}...')
    historial_planet = entrenar_planet(
        modelo_planet, dl_train, dl_val, CONFIG, BEST_CKPT
    )


In [ ]:
if historial_planet is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(historial_planet['train_loss'], color='#2e7d32')
    ax1.set_title('Perdida de entrenamiento')
    ax1.set_xlabel('Epoca') ; ax1.set_ylabel('BCE Loss') ; ax1.grid(alpha=0.3)
    ax2.plot(historial_planet['val_f2'], color='#1565c0')
    ax2.set_title('F2-score en validacion')
    ax2.set_xlabel('Epoca') ; ax2.set_ylabel('F2') ; ax2.grid(alpha=0.3)
    plt.suptitle('Entrenamiento Fase 1 — Planet Amazon', fontsize=12)
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / 'curvas_fase1.png', dpi=120)
    plt.show()


---
## 7. Fine-tuning — Fase 2: Colombia

El backbone preentrenado se integra como encoder de una U-Net. El decoder se entrena desde cero con learning rates diferenciales: el encoder recibe un rate diez veces menor para preservar las representaciones aprendidas en Fase 1.

La pérdida combina Dice Loss y BCE, robusta ante el desbalance espacial típico donde la mayoría de los píxeles son bosque intacto.

In [ ]:
import segmentation_models_pytorch as smp

def construir_unet_colombia(ruta_ckpt_planet=None):
    modelo = smp.Unet(
        encoder_name='efficientnet-b4',
        encoder_weights='imagenet',
        in_channels=4,
        classes=1,
        activation=None
    )
    if ruta_ckpt_planet and Path(ruta_ckpt_planet).exists():
        ckpt   = torch.load(ruta_ckpt_planet, map_location='cpu')
        estado = modelo.state_dict()
        n = 0
        for k, v in ckpt['model_state_dict'].items():
            k2 = k.replace('backbone.', 'encoder.')
            if k2 in estado and estado[k2].shape == v.shape:
                estado[k2] = v ; n += 1
        modelo.load_state_dict(estado)
        print(f'Pesos transferidos desde Fase 1: {n} capas')
    return modelo


def entrenar_colombia(modelo, dl_train, dl_val, config, ruta_ckpt):
    criterio = (smp.losses.DiceLoss(mode='binary') +
                smp.losses.SoftBCEWithLogitsLoss())
    opt = AdamW([
        {'params': modelo.encoder.parameters(),
         'lr': config['colombia_lr'] * 0.1},
        {'params': modelo.decoder.parameters(),
         'lr': config['colombia_lr']},
        {'params': modelo.segmentation_head.parameters(),
         'lr': config['colombia_lr']},
    ], weight_decay=1e-4)

    mejor_iou = 0.0
    historial  = {'train_loss': [], 'val_iou': []}

    for epoca in range(config['colombia_epochs']):
        modelo.train()
        loss_total = 0.0
        for imgs, masks in dl_train:
            imgs  = imgs.to(DEVICE)
            masks = masks.float().unsqueeze(1).to(DEVICE)
            opt.zero_grad()
            loss = criterio(modelo(imgs), masks)
            loss.backward()
            opt.step()
            loss_total += loss.item()

        modelo.eval()
        ious = []
        with torch.no_grad():
            for imgs, masks in dl_val:
                preds = torch.sigmoid(modelo(imgs.to(DEVICE))) > 0.5
                masks = masks.unsqueeze(1).bool()
                inter = (preds.cpu() & masks).float().sum()
                union = (preds.cpu() | masks).float().sum()
                ious.append((inter / (union + 1e-8)).item())

        iou_m  = float(np.mean(ious))
        loss_m = loss_total / len(dl_train)
        historial['train_loss'].append(loss_m)
        historial['val_iou'].append(iou_m)
        print(f'Epoca {epoca+1:>3}/{config["colombia_epochs"]}  '
              f'loss={loss_m:.4f}  IoU={iou_m:.4f}')

        if iou_m > mejor_iou:
            mejor_iou = iou_m
            torch.save({'epoch': epoca,
                        'model_state_dict': modelo.state_dict(),
                        'iou': mejor_iou}, ruta_ckpt)

    return historial


In [ ]:
tiles_disp = list((PROC_COLOMBIA / 'images').rglob('*.npy'))

if len(tiles_disp) == 0:
    print('Pendiente: completar Secciones 2, 3 y 4 antes del fine-tuning.')

elif COL_CKPT.exists():
    modelo_colombia = construir_unet_colombia(BEST_CKPT).to(DEVICE)
    ckpt = torch.load(COL_CKPT, map_location=DEVICE)
    modelo_colombia.load_state_dict(ckpt['model_state_dict'])
    print(f'Checkpoint Colombia cargado — IoU={ckpt["iou"]:.4f}  epoca={ckpt["epoch"]+1}')
    historial_colombia = None

else:
    ds_col_train = ColombiaDataset(PROC_COLOMBIA, 'train')
    ds_col_val   = ColombiaDataset(PROC_COLOMBIA, 'val')
    dl_col_train = DataLoader(ds_col_train, batch_size=16, shuffle=True,
                              num_workers=CONFIG['num_workers'],
                              pin_memory=torch.cuda.is_available())
    dl_col_val   = DataLoader(ds_col_val, batch_size=16, shuffle=False,
                              num_workers=CONFIG['num_workers'],
                              pin_memory=torch.cuda.is_available())
    modelo_colombia = construir_unet_colombia(BEST_CKPT).to(DEVICE)
    print(f'Iniciando fine-tuning en {DEVICE}...')
    historial_colombia = entrenar_colombia(
        modelo_colombia, dl_col_train, dl_col_val, CONFIG, COL_CKPT
    )


---
## 8. Evaluación y Resultados

Evaluación del modelo final sobre el conjunto de prueba con las métricas definidas en la propuesta: IoU por clase, F1-score y visualización de predicciones sobre imágenes reales.

In [ ]:
from sklearn.metrics import classification_report

def evaluar(modelo, dataloader, umbral=0.5):
    modelo.eval()
    preds_all, masks_all = [], []
    with torch.no_grad():
        for imgs, masks in dataloader:
            logits = modelo(imgs.to(DEVICE))
            preds  = (torch.sigmoid(logits) > umbral).cpu().numpy()
            preds_all.append(preds.flatten())
            masks_all.append(masks.numpy().flatten())

    y_pred = np.concatenate(preds_all)
    y_true = np.concatenate(masks_all)
    print(classification_report(y_true, y_pred,
                                 target_names=['Bosque','Deforestado']))
    for clase, nombre in enumerate(['Bosque','Deforestado']):
        inter = ((y_pred==clase) & (y_true==clase)).sum()
        union = ((y_pred==clase) | (y_true==clase)).sum()
        print(f'IoU {nombre:<15}: {inter/(union+1e-8):.4f}')
    return y_true, y_pred


if 'modelo_colombia' in dir() and len(tiles_disp) > 0:
    ds_col_test = ColombiaDataset(PROC_COLOMBIA, 'test')
    dl_col_test = DataLoader(ds_col_test, batch_size=16, shuffle=False)
    y_true, y_pred = evaluar(modelo_colombia, dl_col_test)
else:
    print('Pendiente: completar el entrenamiento de Fase 2.')


In [ ]:
def visualizar_predicciones(modelo, dataset, n=4):
    modelo.eval()
    indices = np.random.choice(len(dataset), n, replace=False)
    fig, axes = plt.subplots(n, 3, figsize=(12, n*3.5))
    fig.suptitle('Predicciones del modelo — Colombia', fontsize=13)

    for fila, idx in enumerate(indices):
        img_t, mask_t = dataset[idx]
        with torch.no_grad():
            pred = (torch.sigmoid(
                modelo(img_t.unsqueeze(0).to(DEVICE))
            ) > 0.5).cpu().squeeze().numpy()

        mean  = np.array([0.485, 0.456, 0.406])
        std   = np.array([0.229, 0.224, 0.225])
        rgb   = np.clip(img_t.numpy().transpose(1,2,0)[:,:,:3] * std + mean, 0, 1)

        for col, (dato, titulo, cmap) in enumerate(zip(
            [rgb, mask_t.numpy(), pred],
            ['Imagen (RGB)', 'Mascara real', 'Prediccion'],
            [None, 'RdYlGn_r', 'RdYlGn_r']
        )):
            axes[fila,col].imshow(dato, cmap=cmap,
                                  vmin=0 if col>0 else None,
                                  vmax=1 if col>0 else None)
            if fila == 0:
                axes[fila,col].set_title(titulo, fontsize=10)
            axes[fila,col].axis('off')

    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / 'predicciones_colombia.png', dpi=120)
    plt.show()


if 'modelo_colombia' in dir() and len(tiles_disp) > 0:
    visualizar_predicciones(modelo_colombia, ds_col_test, n=4)
else:
    print('Pendiente: completar el entrenamiento de Fase 2.')


---
## 9. Prototipo de Aplicación

Inferencia sobre una imagen GeoTIFF completa mediante procesamiento por parches con promedio en zonas de solapamiento. Genera el mapa de probabilidad, el mapa binario y la estimación de área deforestada en hectáreas. El resultado se exporta como GeoTIFF compatible con QGIS.

In [ ]:
def inferencia(ruta_imagen, modelo, tam_tile=224, stride=112, umbral=0.5):
    transform = A.Compose([
        A.Normalize(mean=[0.485,0.456,0.406,0.5],
                    std=[0.229,0.224,0.225,0.25]),
        ToTensorV2()
    ])
    modelo.eval()

    with rasterio.open(ruta_imagen) as src:
        imagen     = src.read([1,2,3,4]).astype(np.float32)
        perfil     = src.profile
        res_m      = abs(src.transform.a)

    _, H, W     = imagen.shape
    mapa_prob   = np.zeros((H,W), dtype=np.float32)
    mapa_conteo = np.zeros((H,W), dtype=np.float32)

    for fila in range(0, H - tam_tile + 1, stride):
        for col in range(0, W - tam_tile + 1, stride):
            tile = np.clip(
                imagen[:, fila:fila+tam_tile, col:col+tam_tile] / 10000.0, 0, 1
            )
            aug  = transform(image=np.transpose(tile,(1,2,0)))
            with torch.no_grad():
                prob = torch.sigmoid(
                    modelo(aug['image'].unsqueeze(0).to(DEVICE))
                ).cpu().squeeze().numpy()
            mapa_prob[fila:fila+tam_tile, col:col+tam_tile]   += prob
            mapa_conteo[fila:fila+tam_tile, col:col+tam_tile] += 1.0

    mapa_prob   /= np.maximum(mapa_conteo, 1)
    mapa_bin     = (mapa_prob > umbral).astype(np.uint8)
    area_ha      = mapa_bin.sum() * (res_m ** 2) / 10_000
    return mapa_prob, mapa_bin, area_ha, perfil


def guardar_geotiff(mapa, perfil, ruta_salida):
    perfil.update(count=1, dtype=rasterio.uint8)
    with rasterio.open(ruta_salida, 'w', **perfil) as dst:
        dst.write(mapa[np.newaxis, :, :])
    print(f'GeoTIFF guardado: {ruta_salida}')


imagenes_col = list(RAW_COLOMBIA.rglob('s2_*.tif'))

if 'modelo_colombia' not in dir():
    print('Pendiente: completar el entrenamiento de Fase 2.')
elif len(imagenes_col) == 0:
    print('Pendiente: no hay imagenes Sentinel-2 colombianas disponibles.')
else:
    img_prueba = imagenes_col[0]
    print(f'Procesando: {img_prueba.name}')
    mapa_prob, mapa_bin, area_ha, perfil = inferencia(img_prueba, modelo_colombia)
    print(f'Area deforestada estimada: {area_ha:,.0f} hectareas')

    ruta_out = OUTPUTS_DIR / f'prediccion_{img_prueba.stem}.tif'
    guardar_geotiff(mapa_bin, perfil, ruta_out)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    im = ax1.imshow(mapa_prob, cmap='RdYlGn_r', vmin=0, vmax=1)
    ax1.set_title('Probabilidad de deforestacion')
    plt.colorbar(im, ax=ax1, fraction=0.03)
    ax2.imshow(mapa_bin, cmap='RdYlGn_r', vmin=0, vmax=1)
    ax2.set_title(f'Mapa binario  ({area_ha:,.0f} ha deforestadas)')
    for ax in [ax1,ax2]: ax.axis('off')
    plt.suptitle(img_prueba.stem, fontsize=11)
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / f'mapa_{img_prueba.stem}.png', dpi=120)
    plt.show()


---
## Notas de uso

### Estado del pipeline

| Sección | Dependencia | Estado |
|---------|-------------|--------|
| 0. Configuracion | Drive (Colab) o ruta local | Ejecutar cada sesion |
| 1. Planet Amazon | ZIP en `data/raw/planet/` | Listo |
| 2. Sentinel-2 GEE | Cuenta GEE activa | Pendiente |
| 3. Mascaras IDEAM | Rasters descargados | Pendiente |
| 4. Pipeline datos | Secciones 1-3 | Parcial (Planet listo) |
| 5. Modelo | — | Listo |
| 6. Entrenamiento Fase 1 | Dataset Planet | Listo |
| 7. Fine-tuning Colombia | Secciones 2-4 | Pendiente |
| 8. Evaluacion | Seccion 7 | Pendiente |
| 9. Prototipo | Seccion 7 | Pendiente |

### Guardar cambios al repositorio desde cualquier entorno
```python
import subprocess
repo = '/content/dl_deforestation' if EN_COLAB else str(BASE)
subprocess.run(['git', 'config', 'user.email', 'correo@ejemplo.com'], cwd=repo)
subprocess.run(['git', 'config', 'user.name', 'Nombre'], cwd=repo)
subprocess.run(['git', 'add', 'notebooks/proyecto_deforestacion.ipynb'], cwd=repo)
subprocess.run(['git', 'commit', '-m', 'update: descripcion del cambio'], cwd=repo)
subprocess.run(['git', 'push'], cwd=repo)
```

### Agregar un nuevo integrante local
Editar la celda de configuración de la Sección 0 agregando una entrada en `RUTAS_LOCALES`:
```python
'nombre_usuario_so': Path(r'ruta\al\proyecto'),
```
Hacer push del cambio para que todos lo tengan.